# Lesson 8: Routing through a gateway with LiteLLM

*Module 3 · about 15 minutes · API key required · optional local model via Ollama*

In Lesson 6 we built a router by hand so we could see every decision it made. In production, most teams don't hand-write the plumbing. They put an **LLM gateway** between their applications and the model providers, and the routing, fallbacks, cost tracking, and budgets live there.

This lesson uses [LiteLLM](https://docs.litellm.ai/), a widely used open-source gateway, to show what that looks like. It comes in two forms: a Python library (the `Router` class, which we'll use here, because it runs inside the notebook) and a proxy server that exposes the same features over an OpenAI-compatible HTTP API. The concepts carry over to other gateways such as Portkey or Kong's AI gateway.

By the end you should be able to:

1. Call models from different tiers (and different providers) through one interface, with the cost of each call worked out for you.
2. Set up model groups and fallbacks, and explain what fallbacks do and don't protect you from.
3. Put quality-based escalation and rule-based routing on top of a gateway.
4. Route sensitive requests to a local model, and work out when self-hosting pays for itself.


### What a gateway gives you

Without a gateway, every application talks to each provider's SDK directly. Switching models means changing code, each team tracks cost its own way (or doesn't), and a provider outage becomes an incident for every team at once.

A gateway puts one layer in the middle:

```
 your apps  -->  gateway  -->  Anthropic / OpenAI / Google / local models / ...
                   |
                   +-- one API for every model
                   +-- model groups ("floor", "mid", "frontier") instead of hard-coded model ids
                   +-- fallbacks and retries when a model errors or is rate-limited
                   +-- cost tracking per call, per key, per team
                   +-- budgets and rate limits (Lesson 7)
```

The important idea is the **model group**: applications ask for `"floor"` or `"mid"` instead of a specific model id. Which model actually answers is configuration. You can change it, or add a second provider for resilience, without touching the application.


### How these notebooks work

Run the cells in order, top to bottom. Before each code cell there's a short explanation of what it does and what to look at in the output. After the important ones there's a note on how to read what you got. Your numbers won't match mine exactly, because models are non-deterministic and prices change, so the notes describe what to look for rather than quoting fixed values.

A few conventions:

- **In class:** notes are cues for when we run this together. If you're working alone, just read them as a prompt to stop and think.
- Every notebook that spends money ends with a **ledger**: one row per API call and the total you spent.
- The **Check yourself** questions at the end have answers hidden under a click. Try them before you look.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, vendor_tokens, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
LIVE = cfg.live


  Provider : anthropic
  floor    : claude-haiku-4-5
  mid      : claude-sonnet-5
  frontier : claude-opus-5
  Cache    : explicit cache_control; read/write are separate buckets.
Switch with LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env
  Rate card: verified 5 Sep 2026 — re-check before presenting.


That cell reads your `.env`, picks OpenAI or Anthropic depending on which key it finds, and prints the three model tiers the notebook will use (floor, mid, frontier).

If the banner names a provider, the live cells will make real calls. Every lesson costs cents, not dollars. If it says `offline`, all the arithmetic still runs, but cells that need a model's answer print a placeholder and tell you they can't draw a conclusion. You can read an offline run, but it's no substitute for a live one in the caching, compression, and routing lessons.

To switch vendors, set `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and run the cell again.


---
## 1. One interface, with cost worked out for you

First we tell LiteLLM what each model costs. LiteLLM ships with its own price list, but we register the course rate card (`coursekit.PRICES`) so that every number in this notebook matches Lessons 1–7. In production you'd normally rely on LiteLLM's built-in prices and only override the ones you've negotiated.

Then we define three **model groups** that point at the three tiers your key can use. For Anthropic we also switch off extended thinking, as `coursekit` does, so small answers aren't cut short.

LiteLLM prints a lot by default, so we quieten it.


In [2]:
import os, re, json, time, warnings, urllib.request
os.environ["LITELLM_LOG"] = "ERROR"
warnings.filterwarnings("ignore")

import litellm
from litellm import Router
litellm.suppress_debug_info = True

PROVIDER = cfg.provider if LIVE else "anthropic"

# Register the course rate card so LiteLLM's cost numbers match the rest of the course.
for model_id, p in PRICES.items():
    for key in (model_id, f"{PROVIDER}/{model_id}"):
        litellm.register_model({key: {
            "input_cost_per_token": p["inp"] / 1e6,
            "output_cost_per_token": p["out"] / 1e6,
            "litellm_provider": PROVIDER, "mode": "chat",
        }})

EXTRA = {"thinking": {"type": "disabled"}} if PROVIDER == "anthropic" else {}

def deployment(group, model_id, **params):
    return {"model_name": group, "litellm_params": {"model": f"{PROVIDER}/{model_id}", **EXTRA, **params}}

MODEL_LIST = [
    deployment("floor", MODELS.floor),
    deployment("mid", MODELS.mid),
    deployment("frontier", MODELS.frontier),
]
router = Router(model_list=MODEL_LIST, num_retries=0)

def call(group, prompt, max_tokens=150, label=None, router=router):
    """Call a model group through the router; record usage and cost in the course ledger."""
    if not LIVE:
        print(f"(offline: would call group '{group}')")
        return None
    resp = router.completion(model=group, messages=[{"role": "user", "content": prompt}],
                             max_tokens=max_tokens)
    cost_usd = resp._hidden_params.get("response_cost") or 0.0
    u = resp.usage
    LEDGER_ROWS.append(dict(label=label or group, group=group, model=resp.model,
                            input=u.prompt_tokens, output=u.completion_tokens, usd=cost_usd))
    return resp

LEDGER_ROWS = []
print("groups:", {d["model_name"]: d["litellm_params"]["model"] for d in MODEL_LIST})


groups: {'floor': 'anthropic/claude-haiku-4-5', 'mid': 'anthropic/claude-sonnet-5', 'frontier': 'anthropic/claude-opus-5'}


Now the same question through each group. The application code is identical each time; only the group name changes. Look at the `usd` column: LiteLLM computed it from the usage the provider returned and the prices we registered.


In [3]:
Q = "In one sentence: what is a model fallback in an LLM gateway?"
for group in ["floor", "mid", "frontier"]:
    resp = call(group, Q, label=f"same question on {group}")
    if resp:
        print(f"{group:<9} {resp.model:<28} {resp.choices[0].message.content[:90]!r}")

if LEDGER_ROWS:
    show(pd.DataFrame(LEDGER_ROWS).style.format({"usd": "${:,.6f}"}))


floor     claude-haiku-4-5-20251001    'A model fallback in an LLM gateway is an automatic switching mechanism that routes request'


mid       claude-sonnet-5              'A model fallback in an LLM gateway is a mechanism that automatically routes a request to a'


frontier  claude-opus-5                'A model fallback in an LLM gateway is an automatic failover mechanism that reroutes a requ'


,label,group,model,input,output,usd
0,same question on floor,floor,claude-haiku-4-5-20251001,24,40,$0.000224
1,same question on mid,mid,claude-sonnet-5,28,67,$0.000726
2,same question on frontier,frontier,claude-opus-5,28,86,$0.002290


---
## 2. Fallbacks: what they're for, and what they aren't

A **fallback** tells the gateway: if a call to this group fails, try that group instead. To see it work, we'll add a group called `primary` whose only deployment points at a model id that doesn't exist, standing in for a model that's been retired, a provider outage, or an exhausted rate limit. Its fallback is `mid`.

Watch which model actually answers.


In [4]:
router_fb = Router(
    model_list=MODEL_LIST + [deployment("primary", "model-that-was-retired-last-week")],
    fallbacks=[{"primary": ["mid"]}],
    num_retries=0,
)
resp = call("primary", "Reply with the single word OK.", max_tokens=10,
            label="primary (broken) -> fallback", router=router_fb)
if resp:
    print(f"asked for group 'primary', answered by: {resp.model}")
    print(f"answer: {resp.choices[0].message.content!r}")


asked for group 'primary', answered by: claude-sonnet-5
answer: 'OK'


The application asked for `primary`, the call failed, and the gateway quietly retried on `mid`. The user got an answer, and no application code had to handle the error.

That is exactly what fallbacks are for: **errors**. Outages, rate limits (HTTP 429), timeouts, and prompts too long for a model's context window (LiteLLM has a separate `context_window_fallbacks` setting for that case). What fallbacks *don't* do is notice a **bad answer**. If the cheap model replies confidently and wrongly, that's a successful call as far as the gateway is concerned. Quality-based escalation is still your code, and the next section adds it.

One cost note: a fallback to a more expensive group is a silent cost increase. If `primary` is down for a day and everything falls back to the frontier model, your bill for that day follows. Watch the share of traffic served by fallbacks as a metric in its own right.


---
## 3. Escalating on quality, on top of the gateway

This is Lesson 6's cascade in a few lines, now running on model groups instead of hard-coded model ids. The escalation signal is a **validator**: a function that checks the answer in code. Here the task is to turn a customer message into JSON with three specific fields, and the validator checks that the output parses and has those fields with sensible values.

Look at how many tasks each tier settles, and what escalation cost.


In [5]:
MESSAGES = [
    "Order NW-2201 arrived two days late and the box was crushed. I want my money back.",
    "Where is NW-3310? It was due yesterday.",
    "Thanks, NW-4102 arrived early, great service!",
    "NW-5120 hasn't moved in the tracking for a week. This is the third time. Cancel everything.",
]
INSTRUCTION = ('Return only JSON with keys "order_id" (string like NW-1234), '
               '"sentiment" (one of "positive", "neutral", "negative") and '
               '"wants_refund" (true or false). No other text.\n\nMessage: ')

def valid(text):
    try:
        data = json.loads(text.strip().removeprefix("```json").removesuffix("```").strip())
    except (ValueError, AttributeError):
        return False
    return (isinstance(data, dict)
            and re.fullmatch(r"NW-\d{4}", str(data.get("order_id", ""))) is not None
            and data.get("sentiment") in {"positive", "neutral", "negative"}
            and isinstance(data.get("wants_refund"), bool))

rows = []
for msg in MESSAGES:
    path, spent = [], 0.0
    for group in ["floor", "mid", "frontier"]:
        resp = call(group, INSTRUCTION + msg, max_tokens=120, label=f"extract on {group}")
        if resp is None:
            break
        spent += LEDGER_ROWS[-1]["usd"]
        path.append(group)
        if valid(resp.choices[0].message.content):
            break
    rows.append(dict(message=msg[:45] + "...", path=" > ".join(path) or "(offline)",
                     usd=spent, final=resp.choices[0].message.content.strip()[:70] if resp else ""))
show(pd.DataFrame(rows).style.format({"usd": "${:,.6f}"}))


,message,path,usd,final
0,Order NW-2201 arrived two days late and the b...,floor,$0.000294,"```json { ""order_id"": ""NW-2201"", ""sentiment"": ""negative"", ""wants"
1,Where is NW-3310? It was due yesterday....,floor,$0.000283,"```json { ""order_id"": ""NW-3310"", ""sentiment"": ""negative"", ""wants"
2,"Thanks, NW-4102 arrived early, great service!...",floor,$0.000284,"```json { ""order_id"": ""NW-4102"", ""sentiment"": ""positive"", ""wants"
3,NW-5120 hasn't moved in the tracking for a we...,floor,$0.000295,"```json { ""order_id"": ""NW-5120"", ""sentiment"": ""negative"", ""wants"


With a clear instruction, the floor model usually produces valid JSON on the first try, so most tasks never leave the cheapest tier. That's the best case for a cascade: a strict, cheap, deterministic check and a task the small model can do. The same pattern works for anything you can check in code, such as a SQL query that parses, a number within range, or a date in the right format.


---
## 4. Routing before the call: rules

A cascade pays for a cheap attempt first. When you can tell from the request itself that it needs a particular tier, it's cheaper to **route up front** and skip the attempt. The simplest version is a handful of explicit rules, which are easy to read, test, and explain to an auditor.

Our rules, checked in order:

1. Anything containing personal data (here, an email address) goes to the `local` group, if a local model is available. More on that in section 5.
2. Anything about legal matters or large amounts goes to `frontier`.
3. Long inputs go to `mid`.
4. Everything else goes to `floor`.

This cell only *decides*. It doesn't call anything, so it runs offline too.


In [6]:
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")

def choose_group(text, local_available):
    if EMAIL.search(text):
        return ("local" if local_available else "mid"), "contains personal data"
    if re.search(r"\b(legal|lawsuit|contract|regulat\w*)\b", text, re.I) or \
       any(int(x.replace(",", "")) >= 1000 for x in re.findall(r"\$(\d[\d,]*)", text)):
        return "frontier", "legal or high value"
    if ntok(text) > 400:
        return "mid", "long input"
    return "floor", "default"

REQUESTS = [
    "What are your delivery hours on Saturdays?",
    "Customer jane.doe@example.com says order NW-1042 never arrived. Draft a reply.",
    "Our contract says late deliveries over $5,000 incur penalties. Does the NW-7710 delay qualify?",
    "Summarise this complaint: " + "The parcel arrived late and damaged. " * 60,
    "Classify sentiment, one word: 'Arrived early, great service.'",
]
for text in REQUESTS:
    group, why = choose_group(text, local_available=False)
    print(f"{group:<9} {why:<24} {text[:70]!r}")


floor     default                  'What are your delivery hours on Saturdays?'
mid       contains personal data   'Customer jane.doe@example.com says order NW-1042 never arrived. Draft '
frontier  legal or high value      'Our contract says late deliveries over $5,000 incur penalties. Does th'
mid       long input               'Summarise this complaint: The parcel arrived late and damaged. The par'
floor     default                  "Classify sentiment, one word: 'Arrived early, great service.'"


In a real system you'd evaluate a rule set like this the same way as any router: label a sample of real requests with the tier they actually needed, and measure how often the rules pick too low (quality risk) or too high (wasted money).


---
## 5. Local models: privacy and cost

Some requests shouldn't leave your network at all, because they contain customer personal data, health information, or source code you're not allowed to send out. A common pattern is to serve those from a model running on your own hardware, and use the cloud for everything else. With a gateway that's just another model group.

The next cell looks for [Ollama](https://ollama.com/), a popular way to run open models locally, at its default address. If it finds one it adds a `local` group and routes the personal-data request there. If not, it says so and moves on. To try it yourself: install Ollama, run `ollama pull qwen2.5:0.5b` (a very small model, about 400 MB), and re-run this cell. Set `OLLAMA_MODEL` in `.env` to use a different model.


In [7]:
def find_local_model():
    try:
        tags = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1))
    except Exception:
        return None
    names = [m["name"] for m in tags.get("models", [])]
    wanted = os.environ.get("OLLAMA_MODEL")
    return wanted if wanted in names else (names[0] if names else None)

LOCAL_MODEL = find_local_model()
if LOCAL_MODEL is None:
    print("No local Ollama model found at localhost:11434, so the local route is skipped.")
    print("Personal-data requests would go to 'mid' instead, which may not be acceptable in production.")
else:
    router_local = Router(model_list=MODEL_LIST + [{
        "model_name": "local",
        "litellm_params": {"model": f"ollama_chat/{LOCAL_MODEL}", "api_base": "http://localhost:11434"},
    }], num_retries=0)
    text = REQUESTS[1]
    group, why = choose_group(text, local_available=True)
    t0 = time.time()
    resp = router_local.completion(model=group, max_tokens=120,
                                   messages=[{"role": "user", "content": text}])
    print(f"routed to '{group}' ({why}) -> {resp.model}, {time.time() - t0:.1f}s, no API charge")
    print(f"answer: {resp.choices[0].message.content[:300]!r}")


routed to 'local' (contains personal data) -> ollama_chat/qwen2.5:0.5b, 0.7s, no API charge
answer: 'Dear Jane,\n\nI hope this message finds you well. I apologize for any inconvenience caused, but I must inform you that I cannot assist with the delivery of an order NW-1042 or any other specific items.\n\nIf you have not received the order by the agreed-upon delivery date, it is important that you conta'


If the local model ran, look at the quality of its answer. A very small model like this one is fine for simple extraction or classification and noticeably weaker at drafting a good reply. Local doesn't mean free, either. The API bill is zero, but you pay for hardware, electricity, and the people who keep it running.

That makes self-hosting a volume question. The cell below compares the all-in monthly cost of one self-hosted GPU server with the API cost of the same traffic. The \$3,240 a month is the course's 2026 estimate for a single H100: about \$1,440 rental, \$1,500 of operations time, and \$300 of other infrastructure. Change it to your own figures.


In [8]:
SELF_HOST_MONTHLY = 3_240          # USD per month, all-in, for one GPU server
T_IN, T_OUT = 800, 200             # tokens per request

rows = []
for tier in ["floor", "mid", "frontier"]:
    model_id = getattr(MODELS, tier)
    per_request = cost(model_id, inp=T_IN, out=T_OUT)
    rows.append(dict(api_tier=tier, model=model_id, usd_per_request=per_request,
                     break_even_requests_per_month=SELF_HOST_MONTHLY / per_request))
show(pd.DataFrame(rows).style.format({"usd_per_request": "${:,.5f}",
                                      "break_even_requests_per_month": "{:,.0f}"}))


,api_tier,model,usd_per_request,break_even_requests_per_month
0,floor,claude-haiku-4-5,$0.00180,"1,800,000"
1,mid,claude-sonnet-5,$0.00360,"900,000"
2,frontier,claude-opus-5,$0.00900,"360,000"


**Reading the table.** Against the cheapest API tier, you need well over a million requests a month before one self-hosted server breaks even, and that assumes the open model you host is good enough to replace it, and that one server can handle the load. Against the frontier tier the break-even volume is much lower, but a small open model usually isn't a substitute for a frontier model.

So for most teams, the case for local models is **privacy and control first, cost second**. Cost becomes the main reason only at high, steady volume.


---
## 6. From the notebook to a shared gateway

Everything above ran inside this notebook. In an organisation you'd run the LiteLLM **proxy** as a shared service, so that every application and team goes through the same routing, fallbacks, cost tracking, and budgets. The same setup as a proxy config file looks roughly like this:

```yaml
model_list:
  - model_name: floor
    litellm_params: { model: anthropic/claude-haiku-4-5 }
  - model_name: mid
    litellm_params: { model: anthropic/claude-sonnet-5 }
  - model_name: mid                       # a second deployment in the same group,
    litellm_params: { model: openai/gpt-5.6-terra }   # used for load balancing and resilience
  - model_name: frontier
    litellm_params: { model: anthropic/claude-opus-5 }
  - model_name: local
    litellm_params: { model: ollama_chat/qwen2.5:0.5b, api_base: "http://gpu-box:11434" }

router_settings:
  fallbacks: [{ "floor": ["mid"] }, { "mid": ["frontier"] }]
  num_retries: 2

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY
  database_url: os.environ/DATABASE_URL   # needed for budgets and spend tracking (Lesson 7)
```

Each team then gets a *virtual key* with its own budget and reset period, and every call is logged with its cost. That's the Track, Attribute, and Control steps from Lesson 7, in one place.


In [9]:
ledger_df = pd.DataFrame(LEDGER_ROWS)
if ledger_df.empty:
    print("no live calls (offline)")
else:
    show(ledger_df.style.format({"usd": "${:,.6f}"}))
    print(f"\nTOTAL SPENT IN THIS NOTEBOOK: {usd(ledger_df.usd.sum())}")


,label,group,model,input,output,usd
0,same question on floor,floor,claude-haiku-4-5-20251001,24,40,$0.000224
1,same question on mid,mid,claude-sonnet-5,28,67,$0.000726
2,same question on frontier,frontier,claude-opus-5,28,86,$0.002290
3,primary (broken) -> fallback,primary,claude-sonnet-5,16,4,$0.000072
4,extract on floor,floor,claude-haiku-4-5-20251001,84,42,$0.000294
5,extract on floor,floor,claude-haiku-4-5-20251001,73,42,$0.000283
6,extract on floor,floor,claude-haiku-4-5-20251001,74,42,$0.000284
7,extract on floor,floor,claude-haiku-4-5-20251001,85,42,$0.000295



TOTAL SPENT IN THIS NOTEBOOK: $0.004468


---
## What to take away

- A gateway gives you one interface to many models. Applications ask for a *model group*, and which model serves it becomes configuration.
- Fallbacks handle errors (outages, rate limits, context-window overflows), not wrong answers. Quality-based escalation is still your own validator.
- Route up front with rules when the request tells you what it needs, and cascade when you can check the answer cheaply.
- Local models are mainly about privacy and control. On cost they only win at high, steady volume.
- Running the gateway as a shared proxy puts routing, cost tracking, and budgets in one place for every team.


### Check yourself

**1. Your `mid` group falls back to `frontier`. The mid-tier provider has an outage for six hours during peak traffic. What happens to your bill, and what would you monitor?**

<details><summary>Show answer</summary>

Every mid-tier request in that window is served, and billed, by the frontier model, at several times the price. Monitor the share of requests served by a fallback, and alert when it rises. Consider falling back to an equivalent model from another provider rather than a more expensive tier.

</details>

**2. Why doesn't a fallback help when the floor model gives a wrong answer?**

<details><summary>Show answer</summary>

From the gateway's point of view the call succeeded: it got a normal response. Fallbacks trigger on errors, not on answer quality. You need your own check (a validator, an eval score, or a confidence signal) to decide to escalate.

</details>

**3. One self-hosted server costs \$3,240 a month. Your traffic is 300,000 requests a month at \$0.0018 each on the API's cheapest tier. Should you self-host for cost reasons?**

<details><summary>Show answer</summary>

API cost is 300,000 × \$0.0018 = **\$540 a month**, far below \$3,240. On cost alone, no. You'd need about 1.8 million requests a month to break even, and only if the local model's quality is good enough.

</details>


### Try it on your own work

List the model ids hard-coded in your applications and replace them with two or three group names (`floor`, `mid`, `frontier`) behind a gateway or a small wrapper. Then add one fallback to a model from a *different* provider for your most important group, and track what share of traffic it serves.
